In [12]:
# 이 코드는 colab에서 실행하는 경우에만 사용합니다.
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import os
import numpy as np

In [14]:
# 이 코드는 colab에서 실행하는 경우에만 사용합니다.
def maybe_make_mnist_raw(out_root="/content/mnist_raw"):
    # 이미 있으면 스킵
    train_dir = os.path.join(out_root, "training", "0")
    test_dir  = os.path.join(out_root, "testing", "0")
    if os.path.isdir(train_dir) and os.path.isdir(test_dir) and len(os.listdir(train_dir)) > 0:
        print("[OK] mnist_raw already exists:", out_root)
        return out_root

    print("[INFO] Creating mnist_raw at:", out_root)
    os.makedirs(out_root, exist_ok=True)

    # TensorFlow로 다운로드만 사용
    from tensorflow.keras.datasets import mnist
    (x_train, y_train), (x_test, y_test) = mnist.load_data()  # (N,28,28), uint8

    def write_split(split_name, X, y):
        split_root = os.path.join(out_root, split_name)
        for label in range(10):
            os.makedirs(os.path.join(split_root, str(label)), exist_ok=True)

        # 파일 저장
        for i in range(X.shape[0]):
            label = int(y[i])
            fpath = os.path.join(split_root, str(label), f"{i:05d}.raw")
            X[i].astype(np.uint8).ravel().tofile(fpath)

    write_split("training", x_train, y_train)
    write_split("testing",  x_test,  y_test)

    print("[DONE] mnist_raw created.")
    return out_root

maybe_make_mnist_raw("/content/mnist_raw")


[OK] mnist_raw already exists: /content/mnist_raw


'/content/mnist_raw'

In [15]:
class SimpleNeuralNet:
    def __init__(self, input_size, hidden_size, output_size):
        self.W1 = np.random.randn(hidden_size, input_size)
        self.b1 = np.zeros((hidden_size, 1))
        self.W2 = np.random.randn(output_size, hidden_size)
        self.b2 = np.zeros((output_size, 1))

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, a):
        return a * (1 - a)

    def forward(self, x):
        self.z1 = np.dot(self.W1, x) + self.b1
        self.a1 = self.sigmoid(self.z1)
        self.z2 = np.dot(self.W2, self.a1) + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def backward(self, x, y, learning_rate):
        m = x.shape[1]
        dz2 = -(y - self.a2) * self.sigmoid_derivative(self.a2)
        dW2 = (1 / m) * np.dot(dz2, self.a1.T)
        db2 = (1 / m) * np.sum(dz2, axis=1, keepdims=True)
        dz1 = np.dot(self.W2.T, dz2) * self.sigmoid_derivative(self.a1)
        dW1 = (1 / m) * np.dot(dz1, x.T)
        db1 = (1 / m) * np.sum(dz1, axis=1, keepdims=True)
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        loss = np.mean((y - self.a2) ** 2) / 2
        return loss

In [16]:
# 데이터셋 읽어오는 함수
def load_mnist_raw(root_dir, split="training"):
    split_dir = os.path.join(root_dir, split)
    X_list, y_list = [], [] # X: 이미지(784차원), Y: 정답(label)

    print(f"[{split}] 데이터 탐색 시작: {split_dir}")

    # 디렉토리가 없는 경우
    if not os.path.exists(split_dir):
        print(f"[오류] 폴더가 없습니다: {split_dir}")
        return np.array([]), np.array([]), np.array([])

    total_loaded = 0  # 로드된 개수 카운트

    # 클래스별 디렉토리 순회
    for label in range(10):
        class_dir = os.path.join(split_dir, str(label))

        # 디렉토리가 없으면 건너뜀
        if not os.path.isdir(class_dir):
            continue

        files = os.listdir(class_dir)
        for fname in files:
            if not fname.endswith(".raw"):
                continue

            fpath = os.path.join(class_dir, fname)
            try:
                # 파일 전체를 읽음
                with open(fpath, "rb") as f:
                    data = np.frombuffer(f.read(), dtype=np.uint8)

                # 정해진 포맷(28x28)이 아닌 경우 건너뜀
                if data.size != 784:
                    continue

                # 각 화솟값(0~255)을 0~1로 정규화
                arr = data.astype(np.float32) / 255.0
                X_list.append(arr)
                y_list.append(label)

                total_loaded += 1

                # 1000개 단위로 로드 현황 알림
                if total_loaded % 1000 == 0:
                    print(f" -> {total_loaded}개 로드 완료...")

            except Exception:
                continue

    # raw 데이터가 없는 경우
    if len(X_list) == 0:
        raise ValueError(f"'{split}' 폴더에서 .raw 데이터를 찾을 수 없습니다.")

    print(f"[{split}] 총 {total_loaded}개 로드 완료!")

    X = np.stack(X_list, axis=0)    # 입력 샘플
    y_int = np.array(y_list, dtype=np.int64)    # 정답 저장
    Y = np.eye(10, dtype=np.float32)[y_int] # 정수 label을 one-hot 벡터로 변환

    # ex)
    # y_int = [1, 2, 3] -> Y = [[0,1,0,0,0,0,0,0,0,0], [0,0,1,0,0,0,0,0,0,0], [0,0,0,1,0,0,0,0,0,0]]

    return X.T, Y.T, y_int

In [17]:
def iterate_minibatches(X, Y, batch_size=64, shuffle=True, seed=0):
    N = X.shape[1] # 샘플 개수
    idx = np.arange(N) # 샘플 인덱스

    if shuffle:
        rng = np.random.default_rng(seed) # 동일 시드, 동일 결과 출력
        rng.shuffle(idx) # 데이터의 인덱스를 섞음

    # 배치 단위로 데이터를 잘라 반환
    for start in range(0, N, batch_size):
        bidx = idx[start:start + batch_size]
        yield X[:, bidx], Y[:, bidx] # 데이터 샘플 반환

In [18]:
def accuracy_from_net(net, X, y_int):
    batch_size = 1000
    correct_count = 0
    N = X.shape[1]

    for i in range(0, N, batch_size):
        X_batch = X[:, i: i + batch_size]
        y_batch = y_int[i: i + batch_size]
        out = net.forward(X_batch)  # 평가이므로, 순전파만 수행
        pred = np.argmax(out, axis=0)
        correct_count += np.sum(pred == y_batch) # 맞춘 샘플 수 누적

    return float(correct_count / N)

In [19]:
def save_weights(net, filename="weights.npz"):
    print(f"\n[저장] 가중치를 '{filename}'에 저장합니다.")
    np.savez(filename, W1=net.W1, b1=net.b1, W2=net.W2, b2=net.b2) #  W1, b1, W2, b2 저장

In [20]:
def main():
    # 로컬로 실행시 경로
    # current_dir = os.path.dirname(os.path.abspath(__file__))
    # data_root = os.path.join(current_dir, "mnist_raw")

    # colab으로 실행시 경로
    data_root = "/content/mnist_raw"

    print(f"데이터 루트: {data_root}")

    try:
        # training(학습) 디렉토리 내 모든 파일(raw) 순회
        X_train, Y_train, y_train_int = load_mnist_raw(data_root, "training")
        # testing(평가) 디렉토리 내 모든 파일(raw) 순회
        X_test, Y_test, y_test_int = load_mnist_raw(data_root, "testing")
    except ValueError as e: # 경로가 잘못된 경우 및 파일이 없는 경우 예외 처리
        print(f"\n[에러] {e}")
        return

    input_size = 784    # 28x28 이미지
    hidden_size = 128   # 은닉층 개수는 자유롭게 설정 가능
    output_size = 10    # 총 10개(0~9)의 클래스

    # 신경망 구성
    net = SimpleNeuralNet(input_size, hidden_size, output_size)

    # 가중치 초기화 조정
    net.W1 *= 0.01
    net.W2 *= 0.01

    # 설정(자유롭게 변경 가능)
    epochs = 20 # 에폭 수는 자유롭게 설정 가능
    batch_size = 10  # 배치 사이즈
    learning_rate = 0.01 # 학습률
    report_interval = 1000  # 1000 스텝마다 로그 출력

    print(f"\n학습 시작 (총 데이터: {X_train.shape[1]}개, 배치 사이즈: {batch_size})")

    # 에폭 횟수만큼 반복
    for epoch in range(epochs):
        print(f"\n=== Epoch {epoch + 1} 시작 ===")

        # 셔플된 데이터 이터레이터 생성
        iterator = iterate_minibatches(X_train, Y_train, batch_size=batch_size, shuffle=True, seed=epoch)

        # 중간 확인을 위한 임시 변수들
        running_loss = 0.0
        running_correct = 0

        step_count = 0

        # 미니배치 단위 학습
        for xb, yb in iterator:
            step_count += batch_size

            # 학습 수행
            net.forward(xb) # 순전파
            loss = net.backward(xb, yb, learning_rate) # 역전파

            # 중간 현황 보고용 loss 누적
            running_loss += loss

            # 현재 샘플 맞췄는지 확인 (배치사이즈 1이므로 바로 비교)
            # 예측 클래스
            pred = np.argmax(net.a2, axis=0)  # net.a2는 현재 샘플의 출력값
            # 정답 클래스
            target = np.argmax(yb, axis=0) # yb는 현재 샘플의 정답

            # 정답을 맞춘 경우 정답 +1
            running_correct += np.sum(pred == target) # np.sum([True, True, False, True, ...]) → True 개수 = 맞춘 샘플 수
            # if pred == target:
            #     running_correct += 1

            # 1000(report_interval)개 단위로 중간 분석
            if step_count % report_interval == 0:
                avg_loss = running_loss / (report_interval // batch_size)
                avg_acc = running_correct / report_interval

                print(f"[Epoch {epoch + 1}] Step {step_count}/{X_train.shape[1]} | "
                      f"최근 {report_interval}개 평균 Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f}")

                # 다음 분석을 위한 통계 초기화
                running_loss = 0.0
                running_correct = 0

        # 매 에폭 종료 후 test 데이터셋으로 검증
        print(f"--- Epoch {epoch + 1} 검증 중... ---")
        test_acc = accuracy_from_net(net, X_test, y_test_int)
        print(f"Epoch {epoch + 1} 완료 | 최종 Test Acc: {test_acc:.4f}")


    # 로컬 실행시 경로
    # save_path = "weights_final.npz"
    # colab 실행시 경로
    save_path = "/content/drive/MyDrive/weights_final.npz"

    # 학습 완료 후 가중치 최종 저장
    save_weights(net, save_path) # NumPy 압축 배열 포맷(W1, b1, W2, b2 저장)

    rng = np.random.default_rng(0)
    idx = rng.choice(X_test.shape[1], size=10, replace=False)

    out = net.forward(X_test[:, idx])
    pred = np.argmax(out, axis=0)

    print("선택 인덱스:", idx)
    print("예측:", pred)
    print("정답:", y_test_int[idx])

In [21]:
if __name__ == "__main__":
    main()

데이터 루트: /content/mnist_raw
[training] 데이터 탐색 시작: /content/mnist_raw/training
 -> 1000개 로드 완료...
 -> 2000개 로드 완료...
 -> 3000개 로드 완료...
 -> 4000개 로드 완료...
 -> 5000개 로드 완료...
 -> 6000개 로드 완료...
 -> 7000개 로드 완료...
 -> 8000개 로드 완료...
 -> 9000개 로드 완료...
 -> 10000개 로드 완료...
 -> 11000개 로드 완료...
 -> 12000개 로드 완료...
 -> 13000개 로드 완료...
 -> 14000개 로드 완료...
 -> 15000개 로드 완료...
 -> 16000개 로드 완료...
 -> 17000개 로드 완료...
 -> 18000개 로드 완료...
 -> 19000개 로드 완료...
 -> 20000개 로드 완료...
 -> 21000개 로드 완료...
 -> 22000개 로드 완료...
 -> 23000개 로드 완료...
 -> 24000개 로드 완료...
 -> 25000개 로드 완료...
 -> 26000개 로드 완료...
 -> 27000개 로드 완료...
 -> 28000개 로드 완료...
 -> 29000개 로드 완료...
 -> 30000개 로드 완료...
 -> 31000개 로드 완료...
 -> 32000개 로드 완료...
 -> 33000개 로드 완료...
 -> 34000개 로드 완료...
 -> 35000개 로드 완료...
 -> 36000개 로드 완료...
 -> 37000개 로드 완료...
 -> 38000개 로드 완료...
 -> 39000개 로드 완료...
 -> 40000개 로드 완료...
 -> 41000개 로드 완료...
 -> 42000개 로드 완료...
 -> 43000개 로드 완료...
 -> 44000개 로드 완료...
 -> 45000개 로드 완료...
 -> 46000개 로드 완료...
 -> 47000개 로